# AMEX Enterprise Credit Risk Platform
## Notebook 08 — Basel III / IFRS 9 Mapping
### Phase 1 · Problem Statement 1: Credit Scoring / PD Prediction

CRISP-DM stage: **Deployment / Regulatory Reporting**. Notebook 8 of 18. Depends on Notebooks 01 and 05 (reads `project_config.json`, `notebook_05_summary.json`, and Notebook 05's saved champion model + `preprocessing_artifacts.joblib`); Notebook 07's model-risk findings are used opportunistically if present, exactly like Notebook 07 used Notebook 06's output — not required.

**What is genuinely computed here, and what is a stated regulatory assumption.** Basel III capital requirements are computed using the actual, publicly published Basel II/III Internal Ratings-Based (IRB) formula for Qualifying Revolving Retail Exposures (QRRE — the Basel category credit cards fall into), applied live to every holdout customer's real predicted PD from Notebook 05's champion model:

- **MEASURED (live, this run):** each holdout customer's predicted probability of default (PD), scored fresh by the real saved champion model — plus the regulatory correlation parameter for QRRE (R = 0.04, a fixed value set by the Basel framework itself, not estimated).
- **ASSUMPTION (stated, editable):** Loss Given Default (LGD) and Exposure at Default (EAD) per account — the Kaggle AMEX dataset contains no recovery-rate or credit-limit data, so these use commonly-cited unsecured-retail industry benchmarks, clearly labeled and easy to edit to your own institution's real estimates.

IFRS 9 staging in this notebook is a **backtest-style exercise on holdout data where the eventual outcome is already known** — Stage 3 (credit-impaired) uses the real observed `target=1` label, which is honest for illustrating the ECL mechanics but is NOT how production staging works (production staging compares a customer's PD today against their PD at origination, which requires a PD term structure this single-snapshot dataset does not provide — stated plainly in Section 6 below).

**Deliverables:** `capital_adequacy_summary.csv`, `ecl_staging_summary.csv`, `regulatory_disclosure_checklist.csv`, 5 charts, and `Basel_III_IFRS9_Mapping_Report.docx`. This is a worked regulatory-capital methodology for a portfolio credit-risk platform, not a substitute for your institution's actual Basel/IFRS9 model validation and audit sign-off.

**Run the single code cell below, once.** Idempotent — every output file is written to a fixed path and overwritten in place on every re-run.

In [ ]:
# =============================================================================
# SECTION 1: ENVIRONMENT SETUP -- LOAD CONFIG FROM NOTEBOOKS 01, 05 (07 OPTIONAL)
# =============================================================================
import os
import sys
import csv
import json
import warnings
from pathlib import Path
from datetime import datetime, timezone


def _section(title: str) -> None:
    bar = "=" * 78
    print(f"\n{bar}\n{title}\n{bar}")


_section("SECTION 1: Environment Setup -- Load Config From Notebooks 01, 05 (07 Optional)")

PROJECT_ROOT = Path(r"C:\Users\rnand\Downloads\amex-default-prediction\AMEX_Enterprise_Credit_Risk_Platform")
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
CONFIG_PATH = ARTIFACTS_DIR / "project_config.json"
NB04_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_04_summary.json"
NB05_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_05_summary.json"  # preferred -- fallback below if absent
NB07_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_07_summary.json"  # optional -- used opportunistically only

for _p, _fix in [(CONFIG_PATH, "run 01_business_understanding.ipynb first"),
                  (NB04_SUMMARY_PATH, "run 04_feature_engineering.ipynb first")]:
    if not _p.exists():
        raise FileNotFoundError(f"{_p} not found.\nFix: {_fix} -- this notebook reads its outputs.")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    PROJECT_CONFIG = json.load(f)
with open(NB04_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB04_SUMMARY = json.load(f)

PILLAR_DIRS = {k: Path(v) for k, v in PROJECT_CONFIG["pillar_dirs"].items()}
RANDOM_SEED = PROJECT_CONFIG["random_seed"]
_resource_limits = PROJECT_CONFIG.get("resource_limits", {})
WARP_THREAD_COUNT = (
    _resource_limits.get("warp_thread_count")
    or PROJECT_CONFIG.get("warp_thread_count")
    or PROJECT_CONFIG["hardware"].get("logical_cores_detected")
)

MODEL_DEV_DIR = PILLAR_DIRS["model_development"]
MODELS_SUBDIR = MODEL_DEV_DIR / "models"
BASEL_DIR = PILLAR_DIRS["basel_ifrs9"]
BASEL_DIR.mkdir(parents=True, exist_ok=True)

TEST_SPLIT_ENG_PATH = Path(NB04_SUMMARY["output_files"]["test_split_engineered.csv"])
MODEL_COMPARISON_PATH = MODEL_DEV_DIR / "model_comparison.csv"
PREPROCESSING_PATH = MODELS_SUBDIR / "preprocessing_artifacts.joblib"

# --- Same resilient champion-identification pattern used by Notebooks 06, 07
#     and 14: prefer notebook_05_summary.json, fall back to model_comparison.csv. ---
NB05_SUMMARY = None
if NB05_SUMMARY_PATH.exists():
    with open(NB05_SUMMARY_PATH, "r", encoding="utf-8") as f:
        NB05_SUMMARY = json.load(f)
    CHAMPION_NAME = NB05_SUMMARY["champion_model"]
    _champion_source = NB05_SUMMARY_PATH.name
elif MODEL_COMPARISON_PATH.exists():
    with open(MODEL_COMPARISON_PATH, "r", encoding="utf-8", newline="") as _f:
        _cmp_rows = list(csv.DictReader(_f))
    if not _cmp_rows or "model" not in _cmp_rows[0] or "holdout_amex_metric" not in _cmp_rows[0]:
        raise RuntimeError(f"{MODEL_COMPARISON_PATH} exists but is missing expected columns -- "
                            f"cannot identify a champion. Fix: re-run 05_model_development.ipynb.")
    _champion_row = max(_cmp_rows, key=lambda r: float(r["holdout_amex_metric"]))
    CHAMPION_NAME = _champion_row["model"]
    _champion_source = f"{MODEL_COMPARISON_PATH.name} (fallback -- {NB05_SUMMARY_PATH.name} not found)"
else:
    raise FileNotFoundError(f"Neither {NB05_SUMMARY_PATH} nor {MODEL_COMPARISON_PATH} was found.\n"
                             f"Fix: run 05_model_development.ipynb first -- this notebook reuses its saved "
                             f"champion model and preprocessing artifacts.")

NB07_SUMMARY = None
if NB07_SUMMARY_PATH.exists():
    with open(NB07_SUMMARY_PATH, "r", encoding="utf-8") as f:
        NB07_SUMMARY = json.load(f)

CHAMPION_MODEL_PATH = MODELS_SUBDIR / f"{CHAMPION_NAME}.joblib"
for _p in (TEST_SPLIT_ENG_PATH, CHAMPION_MODEL_PATH, PREPROCESSING_PATH):
    if not _p.exists():
        raise FileNotFoundError(f"Required file not found: {_p}\nFix: re-run 05_model_development.ipynb.")

print(f"Champion model          : {CHAMPION_NAME}  (identified from: {_champion_source})")
print(f"Notebook 07 MRM output   : {'found -- will cross-reference risk tier' if NB07_SUMMARY else 'not found -- not required for this notebook'}")
print(f"Basel/IFRS9 outputs will be written under: {BASEL_DIR}")
print("\n\u2705 Section 1 complete.")


# =============================================================================
# SECTION 2: WARP HARDWARE CONFIGURATION, LIBRARY IMPORTS & ADAPTIVE RAM CEILING
# =============================================================================
_section("SECTION 2: WARP Hardware Configuration, Library Imports & Adaptive RAM Ceiling")

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

import logging
logger = logging.getLogger("amex_platform")
logger.setLevel(logging.INFO)
if not logger.handlers:
    _handler = logging.StreamHandler(sys.stdout)
    _handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%H:%M:%S"))
    logger.addHandler(_handler)

missing = []
try:
    import polars as pl
except ImportError:
    missing.append("polars")
try:
    import numpy as np
except ImportError:
    missing.append("numpy")
try:
    import pandas as pd
except ImportError:
    missing.append("pandas")
try:
    import psutil
except ImportError:
    missing.append("psutil")
try:
    from scipy.stats import norm
except ImportError:
    missing.append("scipy")
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
except ImportError:
    missing.append("matplotlib")
try:
    import joblib
except ImportError:
    missing.append("joblib")
try:
    from docx import Document
    from docx.shared import Inches
except ImportError:
    missing.append("python-docx")

if missing:
    raise ImportError(
        "Missing required package(s): " + ", ".join(missing) + "\n"
        "Fix: run this in a terminal, then re-run this cell:\n"
        f"    pip install {' '.join(missing)}"
    )

os.environ["POLARS_MAX_THREADS"] = str(WARP_THREAD_COUNT)


def _rss_gb() -> float:
    return psutil.Process().memory_info().rss / 1e9


_process_start_rss_gb = _rss_gb()

# --- Adaptive live-RAM ceiling, same pattern as Notebooks 07 and 14 -- checked
#     fresh every run against whatever is actually free right now, not a stale
#     snapshot from whenever Notebook 01 last ran. ---
_live_vm = psutil.virtual_memory()
LIVE_AVAILABLE_RAM_BYTES = _live_vm.available
ADAPTIVE_RAM_FRACTION = _resource_limits.get("ram_fraction_cap", 0.90)
MAX_RAM_BYTES = int(LIVE_AVAILABLE_RAM_BYTES * ADAPTIVE_RAM_FRACTION)

print(f"WARP_THREAD_COUNT                : {WARP_THREAD_COUNT}")
print(f"Live available RAM right now      : {LIVE_AVAILABLE_RAM_BYTES / 1e9:.1f} GB")
print(f"Adaptive RAM ceiling (this run)   : {MAX_RAM_BYTES / 1e9:.2f} GB")
print(f"GPU probe: not applicable -- this notebook scores the holdout split once (a single predict_proba "
      f"call), the same order of magnitude of work Notebook 05 already benchmarked; there is no repeated "
      f"inference loop here worth a separate GPU benchmark.")
print(f"\nProcess RSS at Section 2 start: {_process_start_rss_gb:.2f} GB")
print("\n\u2705 Section 2 complete.")


# =============================================================================
# SECTION 3: LOAD CHAMPION MODEL, PREPROCESSING ARTIFACTS & PRIOR RESULTS
# =============================================================================
_section("SECTION 3: Load Champion Model, Preprocessing Artifacts & Prior Results")

import time
_t0 = time.time()
champion_model = joblib.load(CHAMPION_MODEL_PATH)
preprocessing_artifacts = joblib.load(PREPROCESSING_PATH)
print(f"Loaded champion model '{CHAMPION_NAME}' from {CHAMPION_MODEL_PATH} ({time.time() - _t0:.1f}s)")

label_encoders = preprocessing_artifacts["label_encoders"]
feature_medians = preprocessing_artifacts["feature_medians"]
scaler = preprocessing_artifacts["scaler"]
all_feature_cols = preprocessing_artifacts["all_feature_cols"]
categorical_encode_cols = preprocessing_artifacts["categorical_encode_cols"]
numeric_feature_cols = preprocessing_artifacts["numeric_feature_cols"]
champion_uses_scaled = CHAMPION_NAME == "logistic_regression"

print(f"Feature columns loaded  : {len(all_feature_cols)} "
      f"({len(numeric_feature_cols)} numeric + {len(categorical_encode_cols)} categorical)")
if NB07_SUMMARY:
    print(f"Notebook 07 risk tier    : {NB07_SUMMARY.get('risk_tier', 'n/a')}")
print("\n\u2705 Section 3 complete.")


# =============================================================================
# SECTION 4: LOAD HOLDOUT DATA, APPLY SAVED PREPROCESSING, SCORE REAL PD (MEASURED)
# =============================================================================
_section("SECTION 4: Load Holdout Data, Apply Saved Preprocessing, Score Real PD")

# --- Same explicit-schema Polars pattern as Notebooks 05-07. Every PD used in
#     this notebook's Basel/IFRS9 calculations is scored HERE, live, by the
#     real saved champion model on the real holdout split -- never assumed. ---
SPLIT_CSV_SCHEMA = {"customer_ID": pl.Utf8, "target": pl.Int8}
for _c in categorical_encode_cols:
    SPLIT_CSV_SCHEMA[_c] = pl.Utf8
for _c in numeric_feature_cols:
    SPLIT_CSV_SCHEMA[_c] = pl.Float32

_t0 = time.time()
holdout_pl = pl.read_csv(str(TEST_SPLIT_ENG_PATH), schema_overrides=SPLIT_CSV_SCHEMA)
print(f"Loaded test_split_engineered.csv: {holdout_pl.shape[0]:,} x {holdout_pl.shape[1]} ({time.time() - _t0:.1f}s)")

_inf_clean_exprs = [
    pl.when(pl.col(c).is_infinite() | pl.col(c).is_nan()).then(None).otherwise(pl.col(c)).cast(pl.Float32).alias(c)
    for c in numeric_feature_cols
]
holdout_pl = holdout_pl.with_columns(_inf_clean_exprs)
for c in categorical_encode_cols:
    holdout_pl = holdout_pl.with_columns(pl.col(c).cast(pl.Utf8).fill_null("__missing__").alias(c))
    _mapping = {cat: i for i, cat in enumerate(label_encoders[c]["classes"])}
    holdout_pl = holdout_pl.with_columns(pl.col(c).replace_strict(_mapping, default=-1, return_dtype=pl.Int32).alias(c))
_impute_exprs = [pl.col(c).fill_null(feature_medians[c]) for c in numeric_feature_cols]
holdout_pl = holdout_pl.with_columns(_impute_exprs)

X_holdout = holdout_pl.select(all_feature_cols).to_numpy().astype(np.float32, copy=False)
y_holdout = holdout_pl.get_column("target").to_numpy().astype(np.int64, copy=False)
holdout_customer_ids = holdout_pl.get_column("customer_ID").to_numpy()

_train_mean = scaler["mean"]
_train_std = scaler["std"]
X_holdout_scaled = (X_holdout - _train_mean) / _train_std
Xc_holdout = X_holdout_scaled if champion_uses_scaled else X_holdout

_t0 = time.time()
PD_12M = champion_model.predict_proba(Xc_holdout)[:, 1].astype(np.float64)
print(f"Scored real PD for {len(PD_12M):,} holdout customers in {time.time() - _t0:.1f}s "
      f"(MEASURED -- champion model '{CHAMPION_NAME}', this run)")
print(f"PD distribution: min {PD_12M.min():.4f}, mean {PD_12M.mean():.4f}, median {np.median(PD_12M):.4f}, "
      f"max {PD_12M.max():.4f}")
print(f"Real observed holdout default rate: {y_holdout.mean():.4%}")
print(f"Process RSS now: {_rss_gb():.2f} GB")
print("\n\u2705 Section 4 complete.")


# =============================================================================
# SECTION 5: BASEL III REGULATORY PARAMETERS (REAL STANDARD + STATED ASSUMPTIONS)
# =============================================================================
_section("SECTION 5: Basel III Regulatory Parameters")

# --- R is NOT a fitted or assumed value -- it is the fixed asset-correlation
#     parameter the Basel II/III framework itself specifies for Qualifying
#     Revolving Retail Exposures (QRRE), the regulatory category credit cards
#     fall into. The 99.9% confidence level is likewise a fixed regulatory
#     standard, not a choice made by this notebook. LGD and EAD ARE stated
#     ASSUMPTIONS -- the Kaggle dataset has no recovery-rate or credit-limit
#     data, so these use commonly-cited unsecured-retail benchmarks. ---
BASEL_QRRE_CORRELATION_R = 0.04       # REGULATORY STANDARD -- fixed by Basel for QRRE, not estimated
BASEL_CONFIDENCE_LEVEL = 0.999        # REGULATORY STANDARD -- 99.9% one-year confidence, Basel framework
BASEL_MINIMUM_CAPITAL_RATIO = 0.08    # REGULATORY STANDARD -- Basel III Pillar 1 minimum (8% of RWA)
BASEL_CAPITAL_CONSERVATION_BUFFER = 0.025  # REGULATORY STANDARD -- Basel III capital conservation buffer

LGD_ASSUMPTION = 0.45                 # ASSUMPTION -- commonly-cited unsecured-retail LGD benchmark
EAD_PER_ACCOUNT_USD = 5_000           # ASSUMPTION -- illustrative average revolving exposure per account
IFRS9_LIFETIME_PD_MULTIPLIER = 3.0    # ASSUMPTION -- illustrative multi-year cumulative-PD approximation
IFRS9_SICR_PD_MULTIPLE = 3.0          # ASSUMPTION -- stated policy: PD > 3x portfolio average triggers Stage 2

print("REGULATORY STANDARD (fixed by the Basel framework, not chosen by this notebook):")
print(f"  QRRE asset correlation (R)       : {BASEL_QRRE_CORRELATION_R}")
print(f"  Confidence level                 : {BASEL_CONFIDENCE_LEVEL:.1%}")
print(f"  Pillar 1 minimum capital ratio   : {BASEL_MINIMUM_CAPITAL_RATIO:.1%} of RWA")
print(f"  Capital conservation buffer      : {BASEL_CAPITAL_CONSERVATION_BUFFER:.1%} of RWA")
print("\nASSUMPTION (stated, editable -- Kaggle AMEX dataset has no real recovery/exposure data):")
print(f"  Loss Given Default (LGD)         : {LGD_ASSUMPTION:.0%}")
print(f"  Exposure at Default (EAD)/account : ${EAD_PER_ACCOUNT_USD:,}")
print(f"  IFRS9 lifetime PD multiplier      : {IFRS9_LIFETIME_PD_MULTIPLIER}x")
print(f"  IFRS9 Stage-2 SICR PD multiple     : {IFRS9_SICR_PD_MULTIPLE}x portfolio average PD")
print("\n\u2705 Section 5 complete.")


# =============================================================================
# SECTION 6: BASEL III IRB CAPITAL CALCULATION -- REAL FORMULA, COMPUTED LIVE PER CUSTOMER
# =============================================================================
_section("SECTION 6: Basel III IRB Capital Calculation (QRRE Formula, Computed Live)")

# --- The actual published Basel II/III IRB risk-weight function for QRRE:
#       K = LGD * [ N( sqrt(1/(1-R)) * G(PD) + sqrt(R/(1-R)) * G(0.999) ) - PD ]
#     where N = standard normal CDF, G = standard normal inverse CDF (probit).
#     No maturity adjustment applies to retail exposures under Basel (that
#     factor is corporate/sovereign/bank-only). RWA = K * 12.5 * EAD. This
#     platform does NOT apply the pre-2023 1.06 scaling factor, consistent
#     with the Basel III finalization framework. Computed here for EVERY
#     holdout customer's real, measured PD from Section 4 -- not a portfolio
#     average plugged in once. ---
_pd_clipped = np.clip(PD_12M, 1e-6, 1 - 1e-6)  # G(0) and G(1) are undefined -- clip to a tiny valid range
_g_pd = norm.ppf(_pd_clipped)
_g_999 = norm.ppf(BASEL_CONFIDENCE_LEVEL)
_r, _lgd = BASEL_QRRE_CORRELATION_R, LGD_ASSUMPTION

_inner = np.sqrt(1.0 / (1.0 - _r)) * _g_pd + np.sqrt(_r / (1.0 - _r)) * _g_999
CAPITAL_REQUIREMENT_K = _lgd * (norm.cdf(_inner) - _pd_clipped)
CAPITAL_REQUIREMENT_K = np.clip(CAPITAL_REQUIREMENT_K, 0.0, None)  # K cannot be negative
RISK_WEIGHT_PCT = CAPITAL_REQUIREMENT_K * 12.5 * 100.0
RWA_PER_CUSTOMER = CAPITAL_REQUIREMENT_K * 12.5 * EAD_PER_ACCOUNT_USD

print(f"Computed Basel QRRE capital requirement (K) and risk-weighted assets (RWA) for "
      f"{len(PD_12M):,} holdout customers, using each customer's own real, measured PD.")
print(f"K (capital requirement, fraction of EAD): min {CAPITAL_REQUIREMENT_K.min():.4f}, "
      f"mean {CAPITAL_REQUIREMENT_K.mean():.4f}, max {CAPITAL_REQUIREMENT_K.max():.4f}")
print(f"Risk weight %                           : min {RISK_WEIGHT_PCT.min():.1f}%, "
      f"mean {RISK_WEIGHT_PCT.mean():.1f}%, max {RISK_WEIGHT_PCT.max():.1f}%")
print("\n\u2705 Section 6 complete.")


# =============================================================================
# SECTION 7: PORTFOLIO CAPITAL ADEQUACY SUMMARY
# =============================================================================
_section("SECTION 7: Portfolio Capital Adequacy Summary")

TOTAL_EAD_USD = EAD_PER_ACCOUNT_USD * len(PD_12M)
TOTAL_RWA_USD = float(RWA_PER_CUSTOMER.sum())
AVG_RISK_WEIGHT_PCT = float(RISK_WEIGHT_PCT.mean())
MIN_PILLAR1_CAPITAL_USD = TOTAL_RWA_USD * BASEL_MINIMUM_CAPITAL_RATIO
BUFFER_CAPITAL_USD = TOTAL_RWA_USD * BASEL_CAPITAL_CONSERVATION_BUFFER
TOTAL_REQUIRED_CAPITAL_USD = MIN_PILLAR1_CAPITAL_USD + BUFFER_CAPITAL_USD

capital_summary = {
    "champion_model": CHAMPION_NAME, "holdout_customers": int(len(PD_12M)),
    "total_ead_usd": TOTAL_EAD_USD, "total_rwa_usd": round(TOTAL_RWA_USD, 2),
    "avg_risk_weight_pct": round(AVG_RISK_WEIGHT_PCT, 2),
    "pillar1_minimum_capital_usd": round(MIN_PILLAR1_CAPITAL_USD, 2),
    "capital_conservation_buffer_usd": round(BUFFER_CAPITAL_USD, 2),
    "total_required_capital_usd": round(TOTAL_REQUIRED_CAPITAL_USD, 2),
    "lgd_assumption": LGD_ASSUMPTION, "ead_per_account_usd_assumption": EAD_PER_ACCOUNT_USD,
}
capital_summary_df = pd.DataFrame([capital_summary])
capital_summary_path = BASEL_DIR / "capital_adequacy_summary.csv"
capital_summary_df.to_csv(capital_summary_path, index=False)

print(f"Total EAD (holdout population, ASSUMPTION x count)  : ${TOTAL_EAD_USD:,.0f}")
print(f"Total RWA (computed, live)                          : ${TOTAL_RWA_USD:,.0f}")
print(f"Average risk weight                                 : {AVG_RISK_WEIGHT_PCT:.1f}%")
print(f"Pillar 1 minimum capital (8% of RWA)                : ${MIN_PILLAR1_CAPITAL_USD:,.0f}")
print(f"Capital conservation buffer (2.5% of RWA)            : ${BUFFER_CAPITAL_USD:,.0f}")
print(f"Total required regulatory capital                   : ${TOTAL_REQUIRED_CAPITAL_USD:,.0f}")
print(f"\u2705 Saved -> {capital_summary_path}")
print("\n\u2705 Section 7 complete.")


# =============================================================================
# SECTION 8: IFRS 9 STAGING -- DOCUMENTED RUBRIC (BACKTEST-STYLE ON HOLDOUT DATA)
# =============================================================================
_section("SECTION 8: IFRS 9 Staging")

# --- HONEST LIMITATION, stated plainly: production IFRS9 staging compares a
#     customer's PD TODAY against their PD AT ORIGINATION to detect a
#     "significant increase in credit risk" (SICR) -- that requires a PD term
#     structure this single-snapshot holdout dataset does not provide. This
#     notebook instead runs a backtest-style exercise: Stage 3 (credit-
#     impaired) uses the REAL, already-known observed target=1 outcome on the
#     holdout set (honest -- these customers really did default); Stage 2 uses
#     a stated relative-PD-deterioration heuristic (PD > 3x portfolio average,
#     a common real-world SICR proxy) among the NOT-yet-defaulted population;
#     everything else is Stage 1. Every number that FEEDS this rubric (PD,
#     portfolio average PD, actual outcome) is real and measured -- the rubric
#     itself is a documented, stated policy, same convention as Notebook 07's
#     risk-tiering rubric. ---
_portfolio_avg_pd = float(PD_12M.mean())
_sicr_threshold = _portfolio_avg_pd * IFRS9_SICR_PD_MULTIPLE

ifrs9_stage = np.where(
    y_holdout == 1, 3,
    np.where(PD_12M > _sicr_threshold, 2, 1)
)

stage_counts = pd.Series(ifrs9_stage).value_counts().sort_index()
print(f"Portfolio average PD (this run)      : {_portfolio_avg_pd:.4f}")
print(f"Stage 2 SICR threshold ({IFRS9_SICR_PD_MULTIPLE}x average): {_sicr_threshold:.4f}")
print(f"\nStaging outcome (holdout population, {len(PD_12M):,} customers):")
for stage in (1, 2, 3):
    _n = int(stage_counts.get(stage, 0))
    _label = {1: "Performing, no SICR", 2: "Significant Increase in Credit Risk (SICR)",
               3: "Credit-Impaired (real observed default)"}[stage]
    print(f"  Stage {stage} ({_label}): {_n:,} customers ({_n / len(PD_12M):.2%})")
print("\n\u2705 Section 8 complete.")


# =============================================================================
# SECTION 9: IFRS 9 EXPECTED CREDIT LOSS (ECL) COMPUTATION BY STAGE
# =============================================================================
_section("SECTION 9: IFRS 9 Expected Credit Loss (ECL) Computation by Stage")

# --- Stage 1: 12-month ECL = PD_12m * LGD * EAD. Stage 2: lifetime ECL, PD
#     approximated as min(1, PD_12m * lifetime_multiplier) -- a stated,
#     illustrative approximation (ASSUMPTION), not a fitted survival curve.
#     Stage 3: the loss has already crystallized in this backtest (target=1),
#     so ECL = LGD * EAD directly (PD_lifetime = 1.0), the standard IFRS9
#     treatment for credit-impaired exposures. ---
pd_lifetime = np.clip(PD_12M * IFRS9_LIFETIME_PD_MULTIPLIER, 0.0, 1.0)

ecl_per_customer = np.where(
    ifrs9_stage == 1, PD_12M * LGD_ASSUMPTION * EAD_PER_ACCOUNT_USD,
    np.where(ifrs9_stage == 2, pd_lifetime * LGD_ASSUMPTION * EAD_PER_ACCOUNT_USD,
             LGD_ASSUMPTION * EAD_PER_ACCOUNT_USD)
)

ecl_df = pd.DataFrame({"customer_ID": holdout_customer_ids, "pd_12m": PD_12M, "ifrs9_stage": ifrs9_stage,
                        "ecl_usd": ecl_per_customer})
ecl_staging_summary = ecl_df.groupby("ifrs9_stage").agg(
    n_customers=("customer_ID", "size"), total_ecl_usd=("ecl_usd", "sum"), avg_pd=("pd_12m", "mean"),
).reset_index()
ecl_staging_summary["pct_of_portfolio"] = ecl_staging_summary["n_customers"] / len(PD_12M)
ecl_staging_path = BASEL_DIR / "ecl_staging_summary.csv"
ecl_staging_summary.to_csv(ecl_staging_path, index=False)

TOTAL_ECL_USD = float(ecl_df["ecl_usd"].sum())
print(ecl_staging_summary.to_string(index=False))
print(f"\nTotal portfolio ECL (holdout, this run): ${TOTAL_ECL_USD:,.0f}")
print(f"\u2705 Saved -> {ecl_staging_path}")
print("\n\u2705 Section 9 complete.")


# =============================================================================
# SECTION 10: BASEL VS. IFRS 9 RECONCILIATION (COMPUTED COMPARISON)
# =============================================================================
_section("SECTION 10: Basel vs. IFRS 9 Reconciliation")

# --- A standard regulatory-reporting comparison: Basel Pillar 1 RWA-implied
#     capital is a REGULATORY CAPITAL BUFFER against unexpected loss at 99.9%
#     confidence; IFRS9 ECL is an ACCOUNTING PROVISION for expected loss. They
#     answer different questions and are not meant to equal each other --
#     the gap itself is the standard "unexpected loss" concept. ---
_implied_basel_expected_loss = float((PD_12M * LGD_ASSUMPTION * EAD_PER_ACCOUNT_USD).sum())
_capital_vs_ecl_gap = TOTAL_REQUIRED_CAPITAL_USD - TOTAL_ECL_USD

print(f"Basel Pillar 1 + buffer required capital : ${TOTAL_REQUIRED_CAPITAL_USD:,.0f}  (unexpected-loss buffer, 99.9% confidence)")
print(f"Basel-implied 12-month expected loss      : ${_implied_basel_expected_loss:,.0f}  (PD x LGD x EAD, portfolio-wide, 12-month)")
print(f"IFRS 9 total ECL (staged)                 : ${TOTAL_ECL_USD:,.0f}  (accounting provision, staged per Section 9)")
print(f"Capital-vs-ECL gap                        : ${_capital_vs_ecl_gap:,.0f}  "
      f"({'capital exceeds provisioning -- expected, this is the unexpected-loss buffer' if _capital_vs_ecl_gap > 0 else 'provisioning exceeds capital -- review staging assumptions'})")
print("\n\u2705 Section 10 complete.")


# =============================================================================
# SECTION 11: REGULATORY DOCUMENTATION & DISCLOSURE CHECKLIST
# =============================================================================
_section("SECTION 11: Regulatory Documentation & Disclosure Checklist")

# --- Every status below is derived from real artifacts found/computed this
#     session -- not a template filled with placeholder "Pass" values, same
#     convention as Notebook 07's governance checklist. ---
regulatory_checklist = [
    {"dimension": "Pillar 1 IRB Capital Calculation (QRRE formula)", "status": "Pass",
     "evidence": f"Computed live for {len(PD_12M):,} holdout customers, this run"},
    {"dimension": "PD Model Validation (SR 11-7 / MRM)",
     "status": "Pass" if NB07_SUMMARY else "Not Yet Completed",
     "evidence": f"Notebook 07 risk tier: {NB07_SUMMARY.get('risk_tier')}" if NB07_SUMMARY else "Run 07_model_risk_management.ipynb"},
    {"dimension": "LGD Estimation", "status": "Assumption Only -- Needs Institution-Specific Estimate",
     "evidence": f"Stated benchmark {LGD_ASSUMPTION:.0%}; dataset has no real recovery-rate data"},
    {"dimension": "EAD Estimation", "status": "Assumption Only -- Needs Institution-Specific Estimate",
     "evidence": f"Stated benchmark ${EAD_PER_ACCOUNT_USD:,}/account; dataset has no real credit-limit data"},
    {"dimension": "IFRS 9 Staging Criteria (SICR)", "status": "Illustrative Only -- Needs PD Term Structure",
     "evidence": "Backtest-style proxy using known holdout outcomes -- see Section 8 limitation note"},
    {"dimension": "ECL Model Documentation", "status": "Pass",
     "evidence": f"ecl_staging_summary.csv, this run -- total ECL ${TOTAL_ECL_USD:,.0f}"},
    {"dimension": "Capital Adequacy Reporting", "status": "Pass",
     "evidence": f"capital_adequacy_summary.csv -- total required capital ${TOTAL_REQUIRED_CAPITAL_USD:,.0f}"},
    {"dimension": "Pillar 3 Disclosure Readiness", "status": "Partial -- Needs Institution Sign-off",
     "evidence": "Quantitative tables ready; narrative disclosure requires Compliance review"},
    {"dimension": "Fair-Lending / Disparate-Impact Review", "status": "Pending",
     "evidence": "Deferred to Notebook 07's governance checklist / dedicated fair-lending review"},
]
regulatory_df = pd.DataFrame(regulatory_checklist)
regulatory_path = BASEL_DIR / "regulatory_disclosure_checklist.csv"
regulatory_df.to_csv(regulatory_path, index=False)

print(regulatory_df.to_string(index=False))
print(f"\u2705 Saved -> {regulatory_path}")
print("\n\u2705 Section 11 complete.")


# =============================================================================
# SECTION 12: CHARTS -- 5 CHARTS (TITLES, AXES, LEGENDS, DATA LABELS)
# =============================================================================
_section("SECTION 12: Charts")

PROBLEM_NAME = "Phase 1 \u00b7 Problem 1 -- Credit Scoring / PD Prediction"
VIZ = {"surface": "#fcfcfb", "text_primary": "#0b0b0b", "text_secondary": "#52514e", "grid": "#e3e2dd",
       "cat_blue": "#2a78d6", "cat_red": "#e34948", "cat_green": "#3a9e5f", "cat_amber": "#d99a2b"}


def _style_axes(ax):
    ax.set_facecolor(VIZ["surface"])
    ax.figure.set_facecolor(VIZ["surface"])
    ax.grid(axis="y", color=VIZ["grid"], linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
    for spine in ("left", "bottom"):
        ax.spines[spine].set_color(VIZ["grid"])
    ax.tick_params(colors=VIZ["text_secondary"], labelsize=9)
    ax.title.set_color(VIZ["text_primary"])
    ax.xaxis.label.set_color(VIZ["text_secondary"])
    ax.yaxis.label.set_color(VIZ["text_secondary"])


# Chart 1: PD distribution histogram
fig, ax = plt.subplots(figsize=(8, 5.5), dpi=150)
ax.hist(PD_12M, bins=40, color=VIZ["cat_blue"], zorder=3)
ax.axvline(_portfolio_avg_pd, color=VIZ["cat_red"], linestyle="--", linewidth=1.4, zorder=4,
           label=f"Portfolio avg PD = {_portfolio_avg_pd:.3f}")
_style_axes(ax)
ax.set_xlabel("Predicted PD (12-month)")
ax.set_ylabel("Number of customers")
ax.set_title(f"{PROBLEM_NAME}\nHoldout PD Distribution ({CHAMPION_NAME}, measured)", fontsize=11)
ax.legend(frameon=False)
fig.tight_layout()
chart1_path = BASEL_DIR / "basel_pd_distribution_chart.png"
fig.savefig(chart1_path, dpi=150, facecolor=VIZ["surface"])
plt.show(); plt.close(fig)
print(f"\u2705 Saved -> {chart1_path}")

# Chart 2: Basel risk-weight curve K(PD) with this portfolio's PDs overlaid
_pd_grid = np.linspace(0.0001, 0.9999, 400)
_g_grid = norm.ppf(_pd_grid)
_inner_grid = np.sqrt(1.0 / (1.0 - _r)) * _g_grid + np.sqrt(_r / (1.0 - _r)) * _g_999
_k_grid = np.clip(_lgd * (norm.cdf(_inner_grid) - _pd_grid), 0.0, None)
_rw_grid = _k_grid * 12.5 * 100.0
fig, ax = plt.subplots(figsize=(8, 5.5), dpi=150)
ax.plot(_pd_grid, _rw_grid, color=VIZ["cat_blue"], linewidth=2, zorder=3, label="Basel QRRE risk-weight curve")
_sample_idx = np.random.RandomState(RANDOM_SEED).choice(len(PD_12M), size=min(400, len(PD_12M)), replace=False)
ax.scatter(PD_12M[_sample_idx], RISK_WEIGHT_PCT[_sample_idx], color=VIZ["cat_amber"], s=10, alpha=0.6, zorder=4,
           label=f"This portfolio's holdout customers (n={len(_sample_idx)} sampled)")
_style_axes(ax)
ax.set_xlabel("PD (12-month)")
ax.set_ylabel("Risk weight (%)")
ax.set_title(f"{PROBLEM_NAME}\nBasel III QRRE Risk-Weight Function (R={_r}, LGD={_lgd:.0%})", fontsize=11)
ax.legend(frameon=False, loc="upper left")
fig.tight_layout()
chart2_path = BASEL_DIR / "basel_risk_weight_curve_chart.png"
fig.savefig(chart2_path, dpi=150, facecolor=VIZ["surface"])
plt.show(); plt.close(fig)
print(f"\u2705 Saved -> {chart2_path}")

# Chart 3: ECL by stage
fig, ax = plt.subplots(figsize=(7, 5.5), dpi=150)
_bars = ax.bar([f"Stage {s}" for s in ecl_staging_summary["ifrs9_stage"]], ecl_staging_summary["total_ecl_usd"],
               color=[VIZ["cat_green"], VIZ["cat_amber"], VIZ["cat_red"]][:len(ecl_staging_summary)], zorder=3)
ax.bar_label(_bars, fmt="$%.0f", padding=3, fontsize=9, color=VIZ["text_primary"])
_style_axes(ax)
ax.set_xlabel("IFRS 9 Stage")
ax.set_ylabel("Total ECL (USD)")
ax.set_title(f"{PROBLEM_NAME}\nIFRS 9 Expected Credit Loss by Stage", fontsize=11)
fig.tight_layout()
chart3_path = BASEL_DIR / "ifrs9_ecl_by_stage_chart.png"
fig.savefig(chart3_path, dpi=150, facecolor=VIZ["surface"])
plt.show(); plt.close(fig)
print(f"\u2705 Saved -> {chart3_path}")

# Chart 4: capital breakdown
fig, ax = plt.subplots(figsize=(6.5, 5.5), dpi=150)
_cap_cats = ["Pillar 1 Minimum\n(8% of RWA)", "Conservation Buffer\n(2.5% of RWA)"]
_cap_vals = [MIN_PILLAR1_CAPITAL_USD, BUFFER_CAPITAL_USD]
_bars = ax.bar(_cap_cats, _cap_vals, color=[VIZ["cat_blue"], VIZ["cat_amber"]], zorder=3)
ax.bar_label(_bars, fmt="$%.0f", padding=3, fontsize=9, color=VIZ["text_primary"])
_style_axes(ax)
ax.set_ylabel("Required capital (USD)")
ax.set_title(f"{PROBLEM_NAME}\nRegulatory Capital Breakdown (Total ${TOTAL_REQUIRED_CAPITAL_USD:,.0f})", fontsize=11)
fig.tight_layout()
chart4_path = BASEL_DIR / "basel_capital_breakdown_chart.png"
fig.savefig(chart4_path, dpi=150, facecolor=VIZ["surface"])
plt.show(); plt.close(fig)
print(f"\u2705 Saved -> {chart4_path}")

# Chart 5: IFRS9 staging distribution
fig, ax = plt.subplots(figsize=(6.5, 5.5), dpi=150)
_stage_labels = [f"Stage {s}\n({int(n):,} customers)" for s, n in zip(ecl_staging_summary["ifrs9_stage"], ecl_staging_summary["n_customers"])]
ax.pie(ecl_staging_summary["n_customers"], labels=_stage_labels,
       colors=[VIZ["cat_green"], VIZ["cat_amber"], VIZ["cat_red"]][:len(ecl_staging_summary)],
       autopct="%1.1f%%", textprops={"fontsize": 9, "color": VIZ["text_primary"]})
ax.set_title(f"{PROBLEM_NAME}\nIFRS 9 Staging Distribution, Holdout Population", fontsize=11)
fig.tight_layout()
chart5_path = BASEL_DIR / "ifrs9_staging_distribution_chart.png"
fig.savefig(chart5_path, dpi=150, facecolor=VIZ["surface"])
plt.show(); plt.close(fig)
print(f"\u2705 Saved -> {chart5_path}")

print("\n\u2705 Section 12 complete.")


# =============================================================================
# SECTION 13: WORD REPORT -- BASEL_III_IFRS9_MAPPING_REPORT.DOCX
# =============================================================================
_section("SECTION 13: Word Report -- Basel_III_IFRS9_Mapping_Report.docx")


def _add_heading(doc, text, level=1):
    return doc.add_heading(text, level=level)


def _add_kv_table(doc, data: dict):
    table = doc.add_table(rows=0, cols=2)
    table.style = "Light Grid Accent 1"
    for k, v in data.items():
        row = table.add_row().cells
        row[0].text = str(k).replace("_", " ").title()
        row[1].text = str(v)
    return table


report = Document()
report.add_heading("AMEX Enterprise Credit Risk Platform", level=0)
report.add_paragraph("Basel III / IFRS 9 Mapping Report -- Notebook 08")
report.add_paragraph(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
report.add_paragraph(
    "This report combines real, measured model output (each holdout customer's predicted PD, scored live "
    "by Notebook 05's champion model) with the actual published Basel II/III IRB formula for Qualifying "
    "Revolving Retail Exposures (a real regulatory formula, not an approximation) and with clearly labeled "
    "ASSUMPTION constants for LGD and EAD, which the Kaggle AMEX dataset does not contain. It is a worked "
    "regulatory-capital methodology, not a substitute for your institution's own Basel/IFRS9 model validation."
)

_add_heading(report, "1. Model & Data Basis (Measured)", level=1)
_add_kv_table(report, {
    "champion_model": CHAMPION_NAME, "holdout_customers_scored": len(PD_12M),
    "measured_portfolio_avg_pd": f"{_portfolio_avg_pd:.4f}",
    "measured_holdout_default_rate": f"{y_holdout.mean():.4%}",
})

_add_heading(report, "2. Basel III Regulatory Parameters", level=1)
report.add_paragraph("Regulatory standard (fixed by Basel, not chosen by this notebook):", style="List Bullet")
_add_kv_table(report, {"qrre_correlation_r": BASEL_QRRE_CORRELATION_R, "confidence_level": f"{BASEL_CONFIDENCE_LEVEL:.1%}",
                        "pillar1_minimum_ratio": f"{BASEL_MINIMUM_CAPITAL_RATIO:.1%}",
                        "capital_conservation_buffer": f"{BASEL_CAPITAL_CONSERVATION_BUFFER:.1%}"})
report.add_paragraph("Stated assumption (editable):", style="List Bullet")
_add_kv_table(report, {"lgd_assumption": f"{LGD_ASSUMPTION:.0%}", "ead_per_account_usd": f"${EAD_PER_ACCOUNT_USD:,}"})

_add_heading(report, "3. Capital Adequacy Summary", level=1)
report.add_picture(str(chart2_path), width=Inches(6.0))
_add_kv_table(report, {k: (f"${v:,.0f}" if "usd" in k else v) for k, v in capital_summary.items()
                        if k not in ("champion_model", "holdout_customers")})
report.add_picture(str(chart4_path), width=Inches(5.5))

_add_heading(report, "4. IFRS 9 Staging", level=1)
report.add_paragraph(
    "IMPORTANT LIMITATION: production IFRS9 staging compares a customer's PD today against their PD at "
    "origination to detect a significant increase in credit risk (SICR) -- this requires a PD term structure "
    "the single-snapshot Kaggle AMEX dataset does not provide. This section instead runs a backtest-style "
    "exercise: Stage 3 uses the real, already-known holdout outcome; Stage 2 uses a stated PD-deterioration "
    "heuristic. This illustrates the ECL mechanics correctly but is not production-ready staging logic."
)
report.add_picture(str(chart5_path), width=Inches(5.0))
_stage_table = report.add_table(rows=1, cols=4)
_stage_table.style = "Light Grid Accent 1"
_hdr = _stage_table.rows[0].cells
_hdr[0].text, _hdr[1].text, _hdr[2].text, _hdr[3].text = "Stage", "Customers", "Avg PD", "Total ECL"
for _, r in ecl_staging_summary.iterrows():
    c = _stage_table.add_row().cells
    c[0].text, c[1].text, c[2].text, c[3].text = str(int(r["ifrs9_stage"])), f"{int(r['n_customers']):,}", \
        f"{r['avg_pd']:.4f}", f"${r['total_ecl_usd']:,.0f}"
report.add_picture(str(chart3_path), width=Inches(6.0))

_add_heading(report, "5. Basel vs. IFRS 9 Reconciliation", level=1)
report.add_paragraph(
    f"Total required regulatory capital (unexpected-loss buffer, 99.9% confidence): "
    f"${TOTAL_REQUIRED_CAPITAL_USD:,.0f}. Total IFRS 9 ECL (accounting provision, expected loss): "
    f"${TOTAL_ECL_USD:,.0f}. These measure different things by design -- capital covers UNEXPECTED loss "
    f"at a high confidence level, while ECL provisions for EXPECTED loss. The gap of "
    f"${_capital_vs_ecl_gap:,.0f} is the expected shape of this comparison."
)

_add_heading(report, "6. Regulatory Documentation & Disclosure Checklist", level=1)
_reg_table = report.add_table(rows=1, cols=3)
_reg_table.style = "Light Grid Accent 1"
_hdr = _reg_table.rows[0].cells
_hdr[0].text, _hdr[1].text, _hdr[2].text = "Dimension", "Status", "Evidence"
for r in regulatory_checklist:
    c = _reg_table.add_row().cells
    c[0].text, c[1].text, c[2].text = r["dimension"], r["status"], r["evidence"]

_add_heading(report, "7. Model Performance Basis", level=1)
report.add_picture(str(chart1_path), width=Inches(6.0))

report_path = BASEL_DIR / "Basel_III_IFRS9_Mapping_Report.docx"
report.save(str(report_path))
print(f"\u2705 Saved -> {report_path}")
print("\n\u2705 Section 13 complete.")


# =============================================================================
# SECTION 14: VERIFICATION -- INTEGRITY CHECKS ON EVERYTHING THIS NOTEBOOK WROTE
# =============================================================================
_section("SECTION 14: Verification")

_checks_passed = True


def _check(label, condition, detail=""):
    global _checks_passed
    if condition:
        print(f"\u2705 {label}")
    else:
        _checks_passed = False
        print(f"\u274c {label}  {detail}")


_check("PD values are within [0,1]", bool(((PD_12M >= 0) & (PD_12M <= 1)).all()))
_check("Capital requirement K is non-negative", bool((CAPITAL_REQUIREMENT_K >= 0).all()))
_check("Risk weights are non-negative", bool((RISK_WEIGHT_PCT >= 0).all()))
_check("IFRS9 stages are all in {1,2,3}", bool(np.isin(ifrs9_stage, [1, 2, 3]).all()))
_check("ECL staging summary covers all stages present in the data",
       set(ecl_staging_summary["ifrs9_stage"]) == set(np.unique(ifrs9_stage)))
_check("Total ECL is non-negative", TOTAL_ECL_USD >= 0)
_check("Total required capital is non-negative", TOTAL_REQUIRED_CAPITAL_USD >= 0)
_check("regulatory checklist covers 9 dimensions", len(regulatory_df) == 9, f"({len(regulatory_df)})")

_expected_files = [capital_summary_path, ecl_staging_path, regulatory_path,
                    chart1_path, chart2_path, chart3_path, chart4_path, chart5_path, report_path]
for fp in _expected_files:
    _check(f"{fp.name} exists and is non-empty", fp.exists() and fp.stat().st_size > 0)

if not _checks_passed:
    raise RuntimeError("One or more Notebook 08 verification checks failed. See \u274c lines above.")

print("\nAll Notebook 08 checks passed.")
print("\n\u2705 Section 14 complete.")


# =============================================================================
# SECTION 15: RESOURCE / PERFORMANCE REPORT
# =============================================================================
_section("SECTION 15: Resource / Performance Report")

_final_rss_gb = _rss_gb()
performance_report = {
    "warp_thread_count_configured": WARP_THREAD_COUNT,
    "adaptive_ram_ceiling_gb": round(MAX_RAM_BYTES / 1e9, 2),
    "live_available_ram_gb_at_start": round(LIVE_AVAILABLE_RAM_BYTES / 1e9, 2),
    "process_rss_at_start_gb": round(_process_start_rss_gb, 2),
    "process_rss_at_end_gb": round(_final_rss_gb, 2),
}
performance_report_path = ARTIFACTS_DIR / "notebook_08_performance_report.json"
with open(performance_report_path, "w", encoding="utf-8") as f:
    json.dump(performance_report, f, indent=2)
print(f"Process RSS: {_process_start_rss_gb:.2f} GB (start) -> {_final_rss_gb:.2f} GB (end)")
print(f"\u2705 Saved -> {performance_report_path}")
print("\n\u2705 Section 15 complete.")


# =============================================================================
# SECTION 16: WRITE NOTEBOOK 08 SUMMARY ARTIFACT (for Notebook 17's rollup)
# =============================================================================
_section("SECTION 16: Write Notebook 08 Summary Artifact")

notebook_08_summary = {
    "notebook": "08_basel_ifrs9_mapping",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "champion_model": CHAMPION_NAME,
    "measured_portfolio_avg_pd": _portfolio_avg_pd,
    "total_rwa_usd": TOTAL_RWA_USD,
    "total_required_capital_usd": TOTAL_REQUIRED_CAPITAL_USD,
    "total_ecl_usd": TOTAL_ECL_USD,
    "ifrs9_stage_counts": {int(s): int(n) for s, n in zip(ecl_staging_summary["ifrs9_stage"], ecl_staging_summary["n_customers"])},
    "lgd_assumption": LGD_ASSUMPTION, "ead_per_account_usd_assumption": EAD_PER_ACCOUNT_USD,
    "output_files": {p.name: str(p) for p in _expected_files + [performance_report_path]},
}
nb08_summary_path = ARTIFACTS_DIR / "notebook_08_summary.json"
with open(nb08_summary_path, "w", encoding="utf-8") as f:
    json.dump(notebook_08_summary, f, indent=2)
print(f"\u2705 Saved -> {nb08_summary_path} (Notebook 17 reads this file to build the rolled-up Basel/IFRS9 section)")
print("\n\u2705 Section 16 complete.")


# =============================================================================
# SECTION 17: COMPLETION SUMMARY
# =============================================================================
_section("SECTION 17: Notebook 08 Complete -- Handoff to Notebook 09")

print("NOTEBOOK 08: BASEL III / IFRS 9 MAPPING -- COMPLETE")
print(f"  Champion (measured PD source)   : {CHAMPION_NAME}")
print(f"  Total RWA (computed)            : ${TOTAL_RWA_USD:,.0f}")
print(f"  Total required capital           : ${TOTAL_REQUIRED_CAPITAL_USD:,.0f}")
print(f"  Total IFRS9 ECL                  : ${TOTAL_ECL_USD:,.0f}")
print(f"  Files produced                   : {len(_expected_files) + 2}")
for _p in _expected_files + [performance_report_path, nb08_summary_path]:
    print(f"    - {_p.name}")
print(f"  Next notebook                    : 09_mlops.ipynb")
print("\n\u2705 Ready to proceed.")
